    # Cattle Average Daily Gain Prediction with a Deep Neural Network

Estimate **average daily gain (ADG, kg/day)** from the `cattle_dataset.csv` dataset.

Target variable: `adg_kg_day`.

# 1. Ambient configuration

## 1.1. Import packages

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch

## 1.2. GPU check

In [11]:
if torch.cuda.is_available():
    print("__CUDNN VERSION:", torch.backends.cudnn.version())
    print("Device Name:", torch.cuda.get_device_name(0))
    device = 'cuda'
else:
    print("CUDA is not available.")
    device = 'cpu'

print('Device:', device)

__CUDNN VERSION: 91501
Device Name: NVIDIA GeForce RTX 5070
Device: cuda


# 2. Exploratory Data Analysis

## 2.1. Load data

In [12]:
data_path = "data/cattle_dataset_2.csv"
target = "adg_kg_day"
leakage = "days_on_pasture"

df = pd.read_csv(data_path)
df

,initial_weight_kg,age_days,sex_male,bos_indicus_proportion_pct,supplement_amount_kg_day,supplement_crude_protein_pct,supplement_metabolizable_energy,supplementation_frequency_days_week,forage_crude_protein_pct,forage_digestibility_pct,days_on_pasture,paddock_rotation,transport_stress,mean_temperature_c,accumulated_rainfall_mm,days_since_health_event,health_event_duration_days,recent_vaccination,recent_deworming,adg_kg_day
0,185,620,0,50.0,0.248000,30,2.0,7,10.42,61.2,381,1,1,29.000000,47.800000,0,0,0,0,0.330709
1,168,620,0,50.0,0.248000,30,2.0,7,10.42,61.2,450,1,1,29.000000,47.800000,0,0,0,0,0.328889
2,153,620,0,50.0,0.248000,30,2.0,7,10.42,61.2,450,1,1,29.000000,47.800000,0,0,0,0,0.317778
3,130,620,0,50.0,0.248000,30,2.0,7,10.42,61.2,450,1,1,29.000000,47.800000,0,0,0,0,0.468889
4,156,620,0,50.0,0.248000,30,2.0,7,10.42,61.2,450,1,1,29.000000,47.800000,0,0,0,0,0.342222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
489,244,430,0,100.0,2.875352,30,20.0,7,5.50,47.2,142,1,0,19.952230,33.004463,0,0,0,0,0.313374
490,222,407,0,100.0,2.886775,30,20.0,7,5.50,47.2,142,1,0,19.877480,26.229234,0,0,0,0,0.331398
491,145,484,0,100.0,0.248000,30,20.0,7,5.50,47.2,302,1,0,21.730790,775.903979,0,0,0,0,0.470800
492,138,476,0,100.0,0.248000,30,20.0,7,5.50,47.2,302,1,0,21.309384,790.922468,0,0,0,0,0.335175


## 2.2. Overview

In [14]:
print(f"Samples: {df.shape[0]}")
print(f"Variables: {df.shape[1]}")

overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unique": df.nunique(),
    "missing": df.isna().sum(),
})
overview

Samples: 494
Variables: 20


,dtype,unique,missing
initial_weight_kg,int64,183,0
age_days,int64,127,0
sex_male,int64,1,0
bos_indicus_proportion_pct,float64,4,0
supplement_amount_kg_day,float64,309,0
supplement_crude_protein_pct,int64,2,0
supplement_metabolizable_energy,float64,3,0
supplementation_frequency_days_week,int64,1,0
forage_crude_protein_pct,float64,6,0
forage_digestibility_pct,float64,6,0
